# 🔬 Visualizador interactivo — Detector PET Hexagonal
Ejecuta las celdas en orden. La última celda lanza los controles interactivos.

In [ ]:
# ── Imports ──────────────────────────────────────────────────
import struct, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize, LogNorm
from matplotlib.cm import ScalarMappable
from scipy.ndimage import gaussian_filter
import ipywidgets as widgets
from IPython.display import display, clear_output

%matplotlib inline
plt.rcParams['figure.facecolor'] = '#0d1117'
plt.rcParams['axes.facecolor']   = '#161b22'
plt.rcParams['axes.edgecolor']   = '#30363d'
plt.rcParams['text.color']       = '#e6edf3'
plt.rcParams['axes.labelcolor']  = '#e6edf3'
plt.rcParams['xtick.color']      = '#e6edf3'
plt.rcParams['ytick.color']      = '#e6edf3'
plt.rcParams['axes.titlecolor']  = '#e6edf3'
plt.rcParams['axes.grid']        = True
plt.rcParams['grid.color']       = '#30363d'
plt.rcParams['grid.linewidth']   = 0.5
print('✓ Librerías cargadas')

✓ Librerías cargadas


In [ ]:
# ── Geometría del detector ────────────────────────────────────
INACTIVE_CH = {1, 16, 18}
ACTIVE_CH   = sorted(set(range(64)) - INACTIVE_CH)   # 61 canales
ICH_TO_IDX  = {ich: i for i, ich in enumerate(ACTIVE_CH)}
IDX_TO_ICH  = {i: ich for ich, i in ICH_TO_IDX.items()}
N_ACTIVE    = len(ACTIVE_CH)  # 61

# Coordenadas axiales del hexágono k=5 → 61 celdas
def hex_grid(k=5):
    R = k - 1
    coords = [(q, r) for q in range(-R, R+1)
              for r in range(max(-R, -q-R), min(R, -q+R)+1)]
    return sorted(coords)

HEX_COORDS  = hex_grid()
ICH_TO_QR   = {ich: HEX_COORDS[i] for i, ich in enumerate(ACTIVE_CH)}
QR_TO_ICH   = {v: k for k, v in ICH_TO_QR.items()}

def qr_to_xy(q, r, size=1.0):
    return size*(np.sqrt(3)*q + np.sqrt(3)/2*r), size*(3/2*r)

SIPM_XY = {ich: qr_to_xy(*ICH_TO_QR[ich]) for ich in ACTIVE_CH}
print(f'✓ Geometría: {N_ACTIVE} SiPMs activos')


✓ Geometría: 61 SiPMs activos


In [ ]:
# ── Lectura del binario ───────────────────────────────────────
def read_dat(path, max_events=200_000):
    """Lee datas#.dat → matriz (N, 61) de cargas brutas."""
    rows, nints = [], []
    with open(path, 'rb') as f:
        while len(rows) < max_events:
            b = f.read(1)
            if not b: break
            nint = struct.unpack('B', b)[0]
            if nint == 0: continue
            row, ok = np.zeros(N_ACTIVE, np.float32), True
            for _ in range(nint):
                rb, ib = f.read(4), f.read(4)
                if len(rb) < 4 or len(ib) < 4: ok = False; break
                rch = struct.unpack('<f', rb)[0]
                ich = struct.unpack('<i', ib)[0]
                if ich in ICH_TO_IDX:
                    row[ICH_TO_IDX[ich]] = rch
            if ok:
                rows.append(row)
                nints.append(nint)
    return np.array(rows, np.float32), np.array(nints, np.int32)
def load_psipm(path):
    """Carga psipm.tsv — sin cabecera, columnas: ich, xsipm, ysipm"""
    for sep in ['\t', ',', ' ', ';']:
        try:
            df = pd.read_csv(path, sep=sep, header=None)
            if df.shape[1] >= 3:
                break
        except:
            continue

    # El archivo no tiene cabecera — asignar nombres directamente
    df = df.iloc[:, :3].copy()
    df.columns = ['ich', 'xsipm', 'ysipm']
    df['ich'] = pd.to_numeric(df['ich'], errors='coerce')
    df = df.dropna(subset=['ich'])
    df['ich'] = df['ich'].astype(int)
    df = df[df['ich'].isin(ACTIVE_CH)].sort_values('ich').reset_index(drop=True)

    print(f"psipm cargado: {len(df)} canales")
    print(f"X rango: [{df['xsipm'].min():.2f}, {df['xsipm'].max():.2f}] mm")
    print(f"Y rango: [{df['ysipm'].min():.2f}, {df['ysipm'].max():.2f}] mm")
    ausentes = [ich for ich in ACTIVE_CH if ich not in df['ich'].values]
    if ausentes:
        print(f"⚠️  Canales ausentes: {ausentes}")
    else:
        print("✓ Los 61 canales están presentes")

    return df.set_index('ich')
def calc_positions(X, psipm=None):
    """Centro de gravedad ponderado por Rch².
    X shape: (N, 61) — columna j corresponde a ACTIVE_CH[j]
    """
    # Construir xs, ys EN EL MISMO ORDEN que ACTIVE_CH
    if psipm is not None:
        xs = np.array([
            float(psipm.loc[ich, 'xsipm']) if ich in psipm.index
            else 0.0
            for ich in ACTIVE_CH
        ])
        ys = np.array([
            float(psipm.loc[ich, 'ysipm']) if ich in psipm.index
            else 0.0
            for ich in ACTIVE_CH
        ])
    else:
        # fallback geométrico — solo si no hay psipm.tsv
        xs = np.array([SIPM_XY[ich][0] for ich in ACTIVE_CH])
        ys = np.array([SIPM_XY[ich][1] for ich in ACTIVE_CH])

    # Verificación de alineación (ejecuta una vez para confirmar)
    # for i, ich in enumerate(ACTIVE_CH):
    #     print(f'idx={i}  Ich={ich}  x={xs[i]:.2f}  y={ys[i]:.2f}')

    r2  = X ** 2                                    # (N, 61)
    rt  = r2.sum(axis=1, keepdims=True).clip(1e-9)  # (N, 1)
    return (r2 * xs).sum(1) / rt[:, 0], (r2 * ys).sum(1) / rt[:, 0]
print('✓ Funciones de lectura listas')

✓ Funciones de lectura listas


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║          CONFIGURA TU DIRECTORIO DE DATOS AQUÍ          ║
# ╚══════════════════════════════════════════════════════════╝

DATA_ROOT = r'E:\Datos TFM'  
# ↑ Cambia esta ruta a la carpeta que contiene Good/ y Bad/

PSIPM_PATH = "E:\Datos TFM\psipm.tsv"   # Ej: r'C:\...\psipm.tsv'  o None para usar geometría axial

# ── Escanear archivos disponibles ────────────────────────────
root = Path(DATA_ROOT)
available = {}
for folder in ['Good\Good', 'Bad\Bad', '.']:
    p = root / folder
    if p.exists():
        files = sorted(p.glob('datas*.dat'))
        if not files: files = sorted(p.glob('*.dat'))
        for f in files:
            label = f'{folder}/{f.name}' if folder != '.' else f.name
            available[label] = f

if available:
    print(f'✓ {len(available)} archivos encontrados:')
    for k in available: print(f'   {k}')
else:
    print('⚠️  No se encontraron archivos .dat — revisa DATA_ROOT')
    print(f'   Ruta actual: {root.resolve()}')

✓ 221 archivos encontrados:
   Good\Good/datas002.dat
   Good\Good/datas003.dat
   Good\Good/datas004.dat
   Good\Good/datas006.dat
   Good\Good/datas007.dat
   Good\Good/datas011.dat
   Good\Good/datas012.dat
   Good\Good/datas013.dat
   Good\Good/datas014.dat
   Good\Good/datas015.dat
   Good\Good/datas018.dat
   Good\Good/datas019.dat
   Good\Good/datas020.dat
   Good\Good/datas021.dat
   Good\Good/datas022.dat
   Good\Good/datas023.dat
   Good\Good/datas026.dat
   Good\Good/datas027.dat
   Good\Good/datas030.dat
   Good\Good/datas031.dat
   Good\Good/datas033.dat
   Good\Good/datas035.dat
   Good\Good/datas036.dat
   Good\Good/datas037.dat
   Good\Good/datas038.dat
   Good\Good/datas039.dat
   Good\Good/datas040.dat
   Good\Good/datas043.dat
   Good\Good/datas044.dat
   Good\Good/datas045.dat
   Good\Good/datas047.dat
   Good\Good/datas048.dat
   Good\Good/datas049.dat
   Good\Good/datas052.dat
   Good\Good/datas054.dat
   Good\Good/datas055.dat
   Good\Good/datas056.dat
   Good\Go

In [ ]:
# ── Cache de datos ya cargados (evita releer el disco) ────────
_cache = {}
psipm  = load_psipm(PSIPM_PATH) if PSIPM_PATH and Path(PSIPM_PATH).exists() else None
if psipm is not None: print(f'✓ psipm.tsv: {len(psipm)} SiPMs')
else:                 print('ℹ️  psipm.tsv no cargado — usando geometría axial')

def get_data(label, max_ev=200_000):
    key = (label, max_ev)
    if key not in _cache:
        print(f'Cargando {label}...', end=' ', flush=True)
        X, ni = read_dat(available[label], max_ev)
        _cache[key] = (X, ni)
        print(f'{len(X):,} eventos')
    return _cache[key]

print('✓ Cache listo')

psipm cargado: 61 canales
X rango: [-13.00, 13.00] mm
Y rango: [-15.00, 15.00] mm
✓ Los 61 canales están presentes
✓ psipm.tsv: 61 SiPMs
✓ Cache listo


In [ ]:
# ── Controles ────────────────────────────────────────────────
w_file   = widgets.Dropdown(options=list(available.keys()), description='Archivo:')
w_file2  = widgets.Dropdown(options=['— ninguno —'] + list(available.keys()), description='Comparar:')
w_maxev  = widgets.SelectionSlider(
    options=[10_000, 50_000, 100_000, 200_000],
    value=50_000, description='Max eventos:', style={'description_width':'120px'})
w_bins   = widgets.IntSlider(value=200, min=50, max=400, step=25, description='Resolución:')
w_smooth = widgets.FloatSlider(value=1.0, min=0, max=5, step=0.5, description='Suavizado:')
w_log    = widgets.Checkbox(value=True, description='Escala log')
w_cmap   = widgets.Dropdown(
    options=['plasma','hot','viridis','magma','inferno','turbo','jet'],
    value='plasma', description='Colores:')
w_emin   = widgets.FloatSlider(value=0,   min=0, max=100, step=1,  description='E mín (%):')
w_emax   = widgets.FloatSlider(value=100, min=0, max=100, step=1,  description='E máx (%):')
w_kill   = widgets.Dropdown(
    options=['— ninguno —'] + [f'Ich={ich}' for ich in ACTIVE_CH],
    description='Apagar SiPM:')
w_btn    = widgets.Button(description='▶ Actualizar', button_style='primary',
                          layout=widgets.Layout(width='160px', height='36px'))
w_out    = widgets.Output()

# ── Layout ───────────────────────────────────────────────────
col1 = widgets.VBox([w_file, w_file2, w_maxev])
col2 = widgets.VBox([w_bins, w_smooth, w_cmap])
col3 = widgets.VBox([w_log, w_emin, w_emax, w_kill])
panel = widgets.HBox([col1, col2, col3],
    layout=widgets.Layout(border='1px solid #30363d', padding='10px', border_radius='8px'))

# ── Diagnóstico de alineamiento (se ejecuta una sola vez al cargar) ──
def check_alignment():
    if psipm is None:
        print('⚠️  psipm no cargado — usando geometría axial, no se puede verificar alineamiento')
        return
    print("── Verificación de alineamiento ACTIVE_CH ↔ psipm ──")
    print(f"{'idx':>4}  {'ACTIVE_CH':>9}  {'en psipm':>8}  {'xsipm':>8}  {'ysipm':>8}  {'ok':>6}")
    all_ok = True
    for i, ich in enumerate(ACTIVE_CH):
        in_psipm = ich in psipm.index
        x = f"{psipm.loc[ich,'xsipm']:.2f}" if in_psipm else 'MISSING'
        y = f"{psipm.loc[ich,'ysipm']:.2f}" if in_psipm else 'MISSING'
        flag = '✓' if in_psipm else '✗ FALTA'
        if not in_psipm:
            all_ok = False
        print(f"{i:>4}  {ich:>9}  {str(in_psipm):>8}  {x:>8}  {y:>8}  {flag}")
    if all_ok:
        print(f"\n✓ Los {len(ACTIVE_CH)} canales están presentes y alineados correctamente")
    else:
        print(f"\n✗ Hay canales de ACTIVE_CH ausentes en psipm — las posiciones usarán x=0, y=0 para ellos")

check_alignment()

# ── Función principal de dibujo ───────────────────────────────
def draw(_=None):
    with w_out:
        clear_output(wait=True)

        if not available:
            print('⚠️  No hay archivos cargados. Revisa DATA_ROOT en la celda anterior.')
            return

        X1, ni1 = get_data(w_file.value, w_maxev.value)
        compare  = w_file2.value != '— ninguno —'
        X2, ni2  = get_data(w_file2.value, w_maxev.value) if compare else (None, None)

        # Filtro de energía por percentil
        def energy_filter(X):
            rcht = X.sum(1)
            lo = np.percentile(rcht[rcht > 0], w_emin.value)
            hi = np.percentile(rcht[rcht > 0], w_emax.value)
            return X[(rcht >= lo) & (rcht <= hi)]

        X1f = energy_filter(X1)
        X2f = energy_filter(X2) if compare else None

        # Apagar canal
        kill_idx = None
        if w_kill.value != '— ninguno —':
            kill_ich = int(w_kill.value.split('=')[1])
            kill_idx = ICH_TO_IDX[kill_ich]
            X1f = X1f.copy(); X1f[:, kill_idx] = 0
            if X2f is not None:
                X2f = X2f.copy(); X2f[:, kill_idx] = 0

        # Calcular posiciones
        px1, py1 = calc_positions(X1f, psipm)
        if compare: px2, py2 = calc_positions(X2f, psipm)

        # Filtrar outliers
        def clip_outliers(px, py, q=99.5):
            lx = np.percentile(np.abs(px), q) * 1.2
            ly = np.percentile(np.abs(py), q) * 1.2
            m  = (np.abs(px) < lx) & (np.abs(py) < ly)
            return px[m], py[m]

        px1, py1 = clip_outliers(px1, py1)
        if compare: px2, py2 = clip_outliers(px2, py2)

        # Extent común
        all_px = np.concatenate([px1, px2]) if compare else px1
        all_py = np.concatenate([py1, py2]) if compare else py1
        ext = [all_px.min(), all_px.max(), all_py.min(), all_py.max()]

        # ── FIGURA ───────────────────────────────────────────
        n_cols = 2 if compare else 1
        fig, axes = plt.subplots(2, n_cols + 1,
            figsize=(7 * (n_cols + 1), 12),
            gridspec_kw={'height_ratios': [2, 1]},
            facecolor='#0d1117')

        t1 = w_file.value
        if kill_idx is not None: t1 += f'  [Ich={kill_ich} OFF]'

        def plot_filling(ax, px, py, title):
            bins = w_bins.value
            h, xe, ye = np.histogram2d(px, py, bins=bins,
                range=[[ext[0], ext[1]], [ext[2], ext[3]]])
            hs = gaussian_filter(h, sigma=w_smooth.value) if w_smooth.value > 0 else h
            vmin = max(0.5, hs[hs > 0].min()) if hs.max() > 0 else 0.5
            norm = LogNorm(vmin=vmin, vmax=hs.max()) if w_log.value else None
            im = ax.imshow(hs.T, origin='lower', aspect='equal',
                extent=[xe[0], xe[-1], ye[0], ye[-1]],
                cmap=w_cmap.value, norm=norm)
            plt.colorbar(im, ax=ax, label='Cuentas', fraction=0.046, pad=0.02)
            ax.set_title(title, pad=8, fontsize=11)
            ax.set_xlabel('X'); ax.set_ylabel('Y')
            ax.text(0.02, 0.97, f'N={len(px):,}', transform=ax.transAxes,
                va='top', fontsize=8, color='#e6edf3')

        if compare:
            plot_filling(axes[0, 0], px1, py1, t1)
            plot_filling(axes[0, 1], px2, py2, w_file2.value)
            ax_diff = axes[0, 2]
            h1, xe, ye = np.histogram2d(px1, py1, bins=w_bins.value,
                range=[[ext[0], ext[1]], [ext[2], ext[3]]])
            h2, _, _   = np.histogram2d(px2, py2, bins=w_bins.value,
                range=[[ext[0], ext[1]], [ext[2], ext[3]]])
            h1n  = h1 / h1.sum() if h1.sum() > 0 else h1
            h2n  = h2 / h2.sum() if h2.sum() > 0 else h2
            diff = h1n - h2n
            lim  = np.abs(diff).max()
            im2  = ax_diff.imshow(diff.T, origin='lower', aspect='equal',
                extent=[xe[0], xe[-1], ye[0], ye[-1]],
                cmap='RdBu_r', vmin=-lim, vmax=lim)
            plt.colorbar(im2, ax=ax_diff, label='Δ (norm.)', fraction=0.046, pad=0.02)
            ax_diff.set_title(f'Diferencia\n{w_file.value} − {w_file2.value}', fontsize=10)
            ax_diff.set_xlabel('X'); ax_diff.set_ylabel('Y')
        else:
            plot_filling(axes[0, 0], px1, py1, t1)
            axes[0, 1].axis('off')

        # ── Espectro de energía ───────────────────────────────
        def plot_energy(ax, X, ni, label, color):
            rcht = X.sum(1)
            rcht = rcht[rcht > 0]
            ax.hist(rcht, bins=120, color=color, alpha=0.7, density=True, label=label)
            ax.axvline(np.median(rcht), color='white', lw=1, ls='--',
                label=f'Med={np.median(rcht):.0f}')
            lo = np.percentile(rcht, w_emin.value)
            hi = np.percentile(rcht, w_emax.value)
            ax.axvspan(lo, hi, color=color, alpha=0.12)
            ax.axvline(lo, color=color, lw=1, ls=':')
            ax.axvline(hi, color=color, lw=1, ls=':')

        ax_e = axes[1, 0]
        plot_energy(ax_e, X1, ni1, w_file.value, '#58a6ff')
        if compare: plot_energy(ax_e, X2, ni2, w_file2.value, '#f78166')
        ax_e.set_title('Espectro de Energía (RchT)')
        ax_e.set_xlabel('Carga Total (RchT)')
        ax_e.set_ylabel('Densidad')
        ax_e.legend(fontsize=8)

        # ── Nint ─────────────────────────────────────────────
        ax_n = axes[1, 1]
        ax_n.hist(ni1, bins=range(0, N_ACTIVE + 2), color='#d2a8ff', alpha=0.75,
            density=True, label=w_file.value, align='left')
        if compare:
            ax_n.hist(ni2, bins=range(0, N_ACTIVE + 2), color='#f78166', alpha=0.55,
                density=True, label=w_file2.value, align='left')
        ax_n.set_title('SiPMs disparados por evento (Nint)')
        ax_n.set_xlabel('Nint'); ax_n.set_ylabel('Densidad')
        ax_n.axvline(ni1.mean(), color='white', lw=1, ls='--',
            label=f'Media={ni1.mean():.1f}')
        ax_n.legend(fontsize=8)

        # ── Mapa de actividad hexagonal ───────────────────────
        ax_h = axes[1, 2] if (n_cols + 1) > 2 else axes[1, 1]
        ax_h.set_facecolor('#0d1117'); ax_h.axis('off')
        mean_ch = X1.mean(0)
        norm_h  = Normalize(vmin=mean_ch.min(), vmax=mean_ch.max())
        cmap_h  = plt.get_cmap('YlOrRd')
        for di in range(N_ACTIVE):
            ich = IDX_TO_ICH[di]
            cx, cy = SIPM_XY[ich]
            killed = (kill_idx is not None and di == kill_idx)
            col  = '#222' if killed else cmap_h(norm_h(mean_ch[di]))
            edge = '#f78166' if killed else '#111'
            lw   = 2.0 if killed else 0.4
            ax_h.add_patch(mpatches.RegularPolygon(
                (cx, cy), numVertices=6, radius=0.85 * 2 / np.sqrt(3),
                orientation=0, facecolor=col, edgecolor=edge, linewidth=lw))
            ax_h.text(cx, cy + 0.05, str(ich), ha='center', va='center',
                fontsize=4.8, color='black' if not killed else '#f78166')
        sm = ScalarMappable(cmap=cmap_h, norm=norm_h)
        sm.set_array([])
        plt.colorbar(sm, ax=ax_h, label='Carga media', fraction=0.046, pad=0.02)
        ax_h.set_xlim(-9, 9); ax_h.set_ylim(-9, 9); ax_h.set_aspect('equal')
        ax_h.set_title('Actividad media por SiPM')

        plt.tight_layout()
        plt.show()


w_btn.on_click(draw)
display(panel, w_btn, w_out)
draw()

── Verificación de alineamiento ACTIVE_CH ↔ psipm ──
 idx  ACTIVE_CH  en psipm     xsipm     ysipm      ok
   0          0      True      3.25     -1.88  ✓
   1          2      True     -0.00      7.50  ✓
   2          3      True      6.50     -7.50  ✓
   3          4      True     -0.00    -15.00  ✓
   4          5      True      9.75     -1.88  ✓
   5          6      True     -0.00      3.75  ✓
   6          7      True      9.75     -5.63  ✓
   7          8      True     13.00     -3.75  ✓
   8          9      True      3.25     -9.38  ✓
   9         10      True      9.75     -9.38  ✓
  10         11      True     -0.00     -3.75  ✓
  11         12      True      6.50     -3.75  ✓
  12         13      True     -0.00     -0.00  ✓
  13         14      True     13.00     -7.50  ✓
  14         15      True     13.00     -0.00  ✓
  15         17      True     -3.25     -5.63  ✓
  16         19      True     -6.50     -7.50  ✓
  17         20      True     -3.25     13.13  ✓
  18       

Button(button_style='primary', description='▶ Actualizar', layout=Layout(height='36px', width='160px'), style=…

Output()

In [ ]:
# Ver el tsv crudo para encontrar el canal que falta
import pandas as pd
from pathlib import Path

df_raw = pd.read_csv(PSIPM_PATH, sep='\t')  # o la sep que funcione
print("Columnas:", df_raw.columns.tolist())
print(f"Filas: {len(df_raw)}")
print()
print(df_raw.to_string())

Columnas: ['37', '-13.0', '-7.5']
Filas: 60

    37  -13.0   -7.5
0   39 -13.00  -3.75
1   45 -13.00  -0.00
2   47 -13.00   3.75
3   49 -13.00   7.50
4   35  -9.75  -9.38
5   34  -9.75  -5.63
6   41  -9.75  -1.88
7   43  -9.75   1.88
8   48  -9.75   5.63
9   46  -9.75   9.38
10  32  -6.50 -11.25
11  19  -6.50  -7.50
12  33  -6.50  -3.75
13  36  -6.50  -0.00
14  42  -6.50   3.75
15  44  -6.50   7.50
16  40  -6.50  11.25
17  21  -3.25 -13.13
18  22  -3.25  -9.38
19  17  -3.25  -5.63
20  31  -3.25  -1.88
21  29  -3.25   1.88
22  28  -3.25   5.63
23  26  -3.25   9.38
24  20  -3.25  13.13
25   4  -0.00 -15.00
26  24  -0.00 -11.25
27  23  -0.00  -7.50
28  11  -0.00  -3.75
29  13  -0.00  -0.00
30   6  -0.00   3.75
31   2  -0.00   7.50
32  30  -0.00  11.25
33  27  -0.00  15.00
34  25   3.25 -13.13
35   9   3.25  -9.38
36  62   3.25  -5.63
37   0   3.25  -1.88
38  61   3.25   1.88
39  59   3.25   5.63
40  58   3.25   9.38
41  57   3.25  13.13
42  38   6.50 -11.25
43   3   6.50  -7.50
44  12   6

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  VISOR DE EVENTO INDIVIDUAL
# ═══════════════════════════════════════════════════════════════

w_ev_file = widgets.Dropdown(options=list(available.keys()), description='Archivo:')
w_ev_idx  = widgets.BoundedIntText(value=0, min=0, max=199_999,
    description='Evento #:', layout=widgets.Layout(width='200px'))
w_ev_btn  = widgets.Button(description='▶ Ver evento', button_style='success',
    layout=widgets.Layout(width='160px', height='36px'))
w_ev_rnd  = widgets.Button(description='🎲 Aleatorio', button_style='',
    layout=widgets.Layout(width='160px', height='36px'))
w_ev_out  = widgets.Output()

def draw_event(_=None):
    with w_ev_out:
        clear_output(wait=True)
        X, _ = get_data(w_ev_file.value, 200_000)
        idx  = w_ev_idx.value
        if idx >= len(X):
            print(f'⚠️  Solo hay {len(X)} eventos'); return
        ev = X[idx]

        fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor='#0d1117')
        fig.suptitle(f'Evento #{idx}  —  {(ev>0).sum()} SiPMs activos  —  '
            f'RchT={ev.sum():.0f}', color='#e6edf3', fontsize=12)

        # Mapa hexagonal
        ax = axes[0]; ax.set_facecolor('#0d1117'); ax.axis('off')
        ax.set_title('Mapa de Carga Rch')
        cmap = plt.get_cmap('hot')
        norm = Normalize(vmin=0, vmax=max(ev.max(), 1))
        for di in range(N_ACTIVE):
            ich = IDX_TO_ICH[di]; cx, cy = SIPM_XY[ich]
            ax.add_patch(mpatches.RegularPolygon(
                (cx,cy), numVertices=6, radius=0.85*2/np.sqrt(3),
                orientation=0, facecolor=cmap(norm(ev[di])),
                edgecolor='#58a6ff' if ev[di]>0 else '#222', linewidth=0.6))
            if ev[di] > 0:
                ax.text(cx,cy+0.05,f'{ev[di]:.0f}',ha='center',va='center',
                    fontsize=4.5,color='white',fontweight='bold')
            else:
                ax.text(cx,cy+0.05,str(ich),ha='center',va='center',
                    fontsize=4.5,color='#444')
        sm = ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
        plt.colorbar(sm, ax=ax, label='Rch', fraction=0.046, pad=0.02)
        ax.set_xlim(-9,9); ax.set_ylim(-9,9); ax.set_aspect('equal')

        # Posición
        ax2 = axes[1]; ax2.set_title('Posición de Interacción')
        xs  = np.array([SIPM_XY[ich][0] for ich in ACTIVE_CH])
        ys  = np.array([SIPM_XY[ich][1] for ich in ACTIVE_CH])
        r2  = ev**2; rt = r2.sum().clip(1e-9)
        px_ev = (r2*xs).sum()/rt; py_ev = (r2*ys).sum()/rt
        sz  = 20 + 300*(ev/ev.max() if ev.max()>0 else ev)
        ax2.scatter(xs, ys, s=sz, c=ev, cmap='hot',
            norm=Normalize(0,ev.max()+1), zorder=2,
            edgecolors='#58a6ff', linewidths=0.4)
        ax2.scatter(px_ev, py_ev, s=250, marker='*', color='#f78166',
            zorder=5, label=f'({px_ev:.2f}, {py_ev:.2f})')
        ax2.scatter(px_ev, py_ev, s=500, marker='o', facecolors='none',
            edgecolors='#f78166', linewidths=1.5, zorder=4)
        rmax = np.hypot(xs,ys).max()*1.15
        ax2.set_xlim(-rmax,rmax); ax2.set_ylim(-rmax,rmax)
        ax2.set_aspect('equal'); ax2.set_xlabel('X'); ax2.set_ylabel('Y')
        ax2.legend(title='Posición estimada', fontsize=9)
        for ich in ACTIVE_CH:
            cx,cy = SIPM_XY[ich]
            ax2.text(cx,cy,str(ich),ha='center',va='center',fontsize=4,color='#555')

        plt.tight_layout(); plt.show()

def random_event(_=None):
    X, _ = get_data(w_ev_file.value, 200_000)
    w_ev_idx.value = int(np.random.randint(0, len(X)))
    draw_event()

w_ev_btn.on_click(draw_event)
w_ev_rnd.on_click(random_event)

display(widgets.HBox([w_ev_file, w_ev_idx, w_ev_btn, w_ev_rnd]), w_ev_out)
random_event()

Output()

Cargando Good\Good/datas002.dat... 

200,000 eventos
